In [1]:
# @title install mrcfile package
%pip install mrcfile

In [11]:
# @title 0. Some package import

import os
import mrcfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import random
import math
import csv
from typing import List, Tuple, Optional
from scipy import stats
from scipy.stats import t
from scipy.spatial  import KDTree
from PIL import Image
import mrcfile

import multiprocessing
import functools
import numba as nb
from tqdm import tqdm

# --- GPU/CPU Parallelism Setup ---
try:
    import cupy as cp
    # Check if a CUDA device is available
    if cp.cuda.runtime.getDeviceCount() > 0:
        CUPY_AVAILABLE = True
        xp = cp
        print("CUDA/CuPy is AVAILABLE. Distance map calculation will use GPU acceleration.")
    else:
        CUPY_AVAILABLE = False
        xp = np
        print("CuPy found, but no CUDA device available. Falling back to NumPy (CPU).")
except:
    CUPY_AVAILABLE = False
    xp = np
    print("CuPy not found. All computations will use NumPy (CPU).")

CUDA/CuPy is AVAILABLE. Distance map calculation will use GPU acceleration.


In [4]:
# @title 1. Mount Google Drive and Set Up File Paths
from google.colab import drive

# Mount your Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


In [5]:
# @title 2. File Paths Set Up
micrograph_denoised = "Topaz Denoise" # @param ["Raw", "Topaz Denoise", "Conv CryoSegNet Denoise Process"]

denoise_prefix = ""
if micrograph_denoised == "Topaz Denoise":
    denoise_prefix = "TpzD_"
if micrograph_denoised == "Conv CryoSegNet Denoise Process":
    denoise_prefix = "C-CSN-D_"

DATA_DIR_PATH = '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017'     # @param {type: "string"}
MICROGRAPH_DIR_PATH = DATA_DIR_PATH + f'/{denoise_prefix}micrographs'
MICROGRAPH_DIR = Path(MICROGRAPH_DIR_PATH)

PARTICLE_COOR_DIR_PATH = DATA_DIR_PATH + '/ground_truth/particle_coordinates'
PARTICLE_COOR_DIR = Path(PARTICLE_COOR_DIR_PATH)

OUTPUT_DIR_PATH = '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017'     # @param {type: "string"}
OUTPUT_DIR_PATH = OUTPUT_DIR_PATH + f'/{denoise_prefix}SNR_analysis'
OUTPUT_DIR = Path(OUTPUT_DIR_PATH)

MASK_DIR_PATH = '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/output/dataset/10017/micrographs_ground_np'     # @param {type: "string"}
MASK_DIR = Path(MASK_DIR_PATH)

!mkdir {OUTPUT_DIR_PATH}

mkdir: cannot create directory ‘/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/TpzD_SNR_analysis’: File exists


---

In [13]:
# @title 3. Parameter set up

block_size = 64                            # @param{type : "integer"}
# Number of blocks take (how many sample for each micrograph)
blocks_num = 10                            # @param{type : "integer"}
particle_filter_block_size = 128           # @param{type : "integer"}
random_state = 42                          # @param{type : "integer"}
# Micrograph edge exclusion (by pixels)
border_size = 100                          # @param{type : "integer"}
min_block_distance = 32                    # @param{type : "integer"}
# Minimum distance required for 2 different blocks
# Maximum number of SNR pairs to compute per micrograph
MAX_SETS_TO_COMPUTE = 20                   # @param{type : "integer"}

EDGE_CLEARANCE_FACTOR = 1.5                # @param{type : "number"}


In [15]:
# @title 4. Functions setup

@dataclass
class SNR_config:
    """
    Configuration for the SNR calculation process.
    All parameters are set to global defaults.
    """
    # Path inputs
    MICROGRAPH_DIR: Path = MICROGRAPH_DIR
    PARTICLE_COOR_DIR: Path = PARTICLE_COOR_DIR
    MASK_DIR: Path = MASK_DIR
    OUTPUT_DIR: Path = OUTPUT_DIR

    # Parameters
    block_size: int = block_size
    blocks_num: int = blocks_num
    PF_block_size: int = particle_filter_block_size
    random_state: Optional[int] = random_state
    border_size: int = border_size
    min_block_distance: int = min_block_distance
    max_sets_to_compute: int = MAX_SETS_TO_COMPUTE

    # List to collect the mean SNR of each micrograph for final statistics
    SNR_list: List[float] = field(default_factory=list)


    def __post_init__(self):
        """Initializes random seed and ensures output directory exists."""
        if self.random_state is not None:
            np.random.seed(self.random_state)
            random.seed(self.random_state)

        self.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



# --- Utility Functions (Data Loading/Saving) ---
def load_mrcfile(cfg: SNR_config) -> List[Tuple[np.ndarray, str]]:
    """Loads all micrograph data by recursively searching for *.mrc files."""

    mrc_files = list(cfg.MICROGRAPH_DIR.rglob('*.mrc'))
    print(f"Found {len(mrc_files)} .mrc files in {cfg.MICROGRAPH_DIR}.")
    micrographs = []

    if not mrc_files:
        raise FileNotFoundError(
            f"No .mrc files found in {cfg.MICROGRAPH_DIR} or its subdirectories. Cannot proceed."
        )

    for mrc_path in mrc_files:
        mrc_filename = mrc_path.name

        try:
            with mrcfile.open(mrc_path) as mrc:
                img_data = mrc.data.astype(np.float64)

            print(f"Successfully loaded {mrc_filename} with shape {img_data.shape}.")
            micrographs.append((img_data, mrc_filename))

        except Exception as e:
            print(f"WARNING: Could not load or read MRC file {mrc_path}. Skipping. Error: {e}")
            continue

    return micrographs

def load_particle_coor(cfg: SNR_config, coor_path: Path) -> np.ndarray:
    """Loads particle coordinates from a CSV file (X-Coordinate, Y-Coordinate)."""

    try:
        with open(coor_path, 'r', newline='') as csvfile:
            reader = csv.reader(csvfile)
            header = next(reader)

            try:
                x_idx = header.index('X-Coordinate')
                y_idx = header.index('Y-Coordinate')
            except ValueError:
                raise ValueError("CSV file header must contain 'X-Coordinate' and 'Y-Coordinate' columns.")

            coords = []
            for row in reader:
                try:
                    x = float(row[x_idx])
                    y = float(row[y_idx])
                    coords.append((x, y))
                except (ValueError, IndexError):
                    continue

        if not coords:
            return np.array([])

        return np.array(coords, dtype=np.float64)

    except FileNotFoundError:
        return np.array([])
    except Exception as e:
        return np.array([])

def load_mask(cfg: SNR_config, mask_path: Path) -> np.ndarray:
    """Loads a PNG mask file, normalizing particles (>0.1) to 1.0 and background to 0.0."""

    try:
        with Image.open(mask_path) as img:
            mask_data = np.array(img.convert('L'), dtype=np.float32)

        mask_data[mask_data > 0.1] = 1.0
        mask_data[mask_data <= 0.1] = 0.0
        return mask_data

    except FileNotFoundError:
        return np.array([])
    except Exception as e:
        return np.array([])

def SNR_csv_write_micrograph_data(cfg: SNR_config, mrc_filename: str, method: str,
                                  mean_micrograph_snr: float, total_pairs_N: int,
                                  file_name: str = "micrograph_snr_results.csv"):
    """Writes the single mean SNR result for a micrograph to a CSV file."""

    output_path = cfg.OUTPUT_DIR / file_name
    file_exists = output_path.exists()

    try:
        with open(output_path, mode='a', newline='') as file:
            writer = csv.writer(file)

            if not file_exists:
                writer.writerow([
                    "Micrograph_Filename",
                    "Method",
                    "Total_Pairs_N",
                    "Mean_SNR_Micrograph"
                ])

            writer.writerow([
                mrc_filename,
                method,
                total_pairs_N,
                f"{mean_micrograph_snr:.6f}"
            ])
    except Exception as e:
        print(f"ERROR: Failed to write to CSV file {output_path}. Error: {e}")


# --- Core SNR Calculation Functions (Numba Optimized) ---

#@nb.njit(cache=True)
@nb.njit
def _extract_block(img_data: np.ndarray, x: int, y: int, size: int) -> np.ndarray:
    """
    Extracts a square block of 'size' x 'size' pixels centered at (x, y).
    Uses Numba for acceleration and precise size logic for even/odd sizes.
    """
    h, w = img_data.shape

    if size % 2 == 0: # Even size (e.g., 64)
        half_size = size // 2
        # Asymmetric offsets
        x_start = max(0, x - half_size)
        x_end = min(w, x + half_size)
        y_start = max(0, y - half_size)
        y_end = min(h, y + half_size)

    else: # Odd size (e.g., 65)
        size_adj = size - 1
        half_size = size_adj // 2
        # Symmetric offsets
        x_start = max(0, x - half_size)
        x_end = min(w, x + half_size + 1)
        y_start = max(0, y - half_size)
        y_end = min(h, y + half_size + 1)

    block = img_data[y_start:y_end, x_start:x_end]
    return block

#@nb.njit(cache=True)
@nb.njit
def SNR_computation(particle_region: np.ndarray, background_region: np.ndarray) -> float:
    """
    Computes the log-variance ratio (DB version of NR).
    Optimized with Numba: log10( Var(Signal) / Var(Noise) ).
    """
    v_b = np.var(background_region)
    mu_b = np.mean(background_region)

    if v_b <= 1e-12:
        return 0.0

    s = particle_region - mu_b
    v_s = np.var(s)

    snr_i = math.log10(v_s / v_b)
    return snr_i


# --- Helper for Mask Updating (Numba Optimized) ---

#@nb.njit(cache=True)
@nb.njit
def _numba_update_masks(pf_binary_mask, masked_img_content, particles_coor, pf_half, pf_half_adj, w, h, mask_value):
    """Numba helper to apply particle masks (1s and NaNs) efficiently."""
    for i in range(particles_coor.shape[0]):
        x, y = int(particles_coor[i, 0]), int(particles_coor[i, 1])

        # Bounding box coordinates based on precise PF block size logic
        x_start = max(0, x - pf_half_adj)
        x_end = min(w, x + pf_half)
        y_start = max(0, y - pf_half)
        y_end = min(h, y + pf_half_adj)

        pf_binary_mask[y_start:y_end, x_start:x_end] = 1
        masked_img_content[y_start:y_end, x_start:x_end] = mask_value


# --- Helper for Distance Transform (CUDA Accelerated) ---

def _calculate_distance_map(pf_binary_mask: np.ndarray) -> np.ndarray:
    """
    Calculates the minimum Euclidean distance from every pixel to the nearest particle pixel (Distance Transform).
    Uses CuPy (xp) if available, falls back to NumPy/KDTree.
    """
    global xp

    mask_data = xp.asarray(pf_binary_mask)

    particle_pixels_y, particle_pixels_x = xp.where(mask_data == 1)
    particle_pixel_coords = xp.vstack((particle_pixels_x, particle_pixels_y)).T

    if particle_pixel_coords.size == 0:
        return np.full(pf_binary_mask.shape, np.inf, dtype=np.float32)

    # KDTree requires NumPy arrays
    if CUPY_AVAILABLE:
        particle_pixel_coords_np = cp.asnumpy(particle_pixel_coords)
    else:
        particle_pixel_coords_np = particle_pixel_coords

    particle_tree = KDTree(particle_pixel_coords_np)

    # Setup Query Coordinates (NumPy)
    h, w = pf_binary_mask.shape
    y_coords, x_coords = np.mgrid[0:h, 0:w]
    all_pixel_coords = np.vstack((x_coords.ravel(), y_coords.ravel())).T

    # Query the KDTree for the distance (CPU operation)
    distances_flat, _ = particle_tree.query(all_pixel_coords, k=1)

    distance_map = distances_flat.reshape(h, w)

    return distance_map


# --- Version 1: Block-Based (Smart Distance Sampling) Logic ---

def _remove_particle_blocks(cfg: SNR_config, np_img_data: np.ndarray, particles_coor: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Creates the binary mask (for distance map) and content mask (for background exclusion)
    based on particle coordinates. Uses Numba helper for speed.
    """
    h, w = np_img_data.shape
    pf_binary_mask = np.zeros((h, w), dtype=np.int8)
    pf_size = cfg.PF_block_size

    # Calculate the precise start and end offsets to ensure N x N mask size.
    if pf_size % 2 == 0: # Even (e.g., 128). Target slice: [x - 64 : x + 64].
        pf_half_start = pf_size // 2      # N/2
        pf_half_end = pf_size // 2        # N/2
    else: # Odd (e.g., 129). Target slice: [x - 64 : x + 65].
        pf_half_start = (pf_size - 1) // 2 # (N-1)/2
        pf_half_end = (pf_size - 1) // 2 + 1 # (N-1)/2 + 1

    masked_img_content = np_img_data.copy()
    mask_value = np.nan

    _numba_update_masks(pf_binary_mask, masked_img_content, particles_coor,
                        pf_half_start, pf_half_end, w, h, mask_value)

    return masked_img_content, pf_binary_mask

def _pick_background_coords(cfg: SNR_config, masked_img_content: np.ndarray, pf_binary_mask: np.ndarray) -> List[Tuple[int, int]]:
    """
    Selects clean background blocks by filtering candidates based on:
    1. Distance Map clearance from particle edges.
    2. Mask Purity (low particle contamination).
    3. Minimum distance between selected background blocks (non-overlap).
    """
    distance_map = _calculate_distance_map(pf_binary_mask)

    size = cfg.block_size
    if size % 2 == 0:
        half_floor = size // 2 - 1
    else:
        half_floor = size // 2

    h, w = masked_img_content.shape
    min_x = cfg.border_size + half_floor
    max_x = w - cfg.border_size - half_floor
    min_y = cfg.border_size + half_floor
    max_y = h - cfg.border_size - half_floor

    if max_x <= min_x or max_y <= min_y:
        return []

    # Filtering parameters
    min_required_clearance = cfg.block_size * EDGE_CLEARANCE_FACTOR
    MAX_MASK_CONTAMINATION = 0.03

    clean_centers = []
    SAMPLE_STEP = 8

    for y in range(min_y, max_y, SAMPLE_STEP):
        for x in range(min_x, max_x, SAMPLE_STEP):

            # 1. Distance Map check
            if distance_map[y, x] > min_required_clearance:

                # 2. Mask Purity check (Numba optimized _extract_block)
                mask_block = _extract_block(pf_binary_mask, x, y, cfg.block_size)

                if np.mean(mask_block) < MAX_MASK_CONTAMINATION:
                    clean_centers.append((x, y))

    if not clean_centers:
        return []

    # 3. Non-Overlap Filtering
    final_coords = []
    rng = np.random.default_rng(cfg.random_state)
    rng.shuffle(clean_centers)

    min_center_dist = cfg.block_size + cfg.min_block_distance

    for x, y in clean_centers:
        if len(final_coords) >= cfg.max_sets_to_compute:
            break

        is_too_close = False
        for px, py in final_coords:
            dist_x = abs(x - px)
            dist_y = abs(y - py)

            if dist_x < min_center_dist and dist_y < min_center_dist:
                is_too_close = True
                break

        if not is_too_close:
            final_coords.append((x, y))

    return final_coords


def SNR_within_blocks_block_based(cfg: SNR_config, np_img_data: np.ndarray, particles_coor: np.ndarray) -> Tuple[float, int]:
    """
    Calculates the mean SNR using the block-based, distance-map guided selection method.
    """
    num_to_sample = cfg.max_sets_to_compute

    if len(particles_coor) < cfg.blocks_num:
        pass

    # 1. Randomly select particle centers
    rng = np.random.default_rng(cfg.random_state)
    num_particles_available = len(particles_coor)

    current_num_sets = min(num_to_sample, num_particles_available)

    if current_num_sets == 0:
        return 0.0, 0

    indices = rng.choice(num_particles_available, size=current_num_sets, replace=False)
    particle_centers = particles_coor[indices]

    # 2. Prepare masks (Numba optimized)
    masked_img_content, pf_binary_mask = _remove_particle_blocks(cfg, np_img_data, particles_coor)

    # 3. Pick background block coordinates
    background_centers_all = _pick_background_coords(cfg, masked_img_content, pf_binary_mask)

    current_num_sets = min(current_num_sets, len(background_centers_all))

    if current_num_sets == 0:
        return 0.0, 0

    # Randomly select the required number of background blocks
    background_centers = random.sample(background_centers_all, current_num_sets)
    particle_centers = particle_centers[:current_num_sets]

    # 4. Extract blocks and compute SNR (Numba optimized loop)
    snr_list = []
    for i in range(current_num_sets):
        px, py = particle_centers[i]
        particle_region = _extract_block(np_img_data, int(px), int(py), cfg.block_size)

        bx, by = background_centers[i]
        background_region = _extract_block(np_img_data, bx, by, cfg.block_size)

        if particle_region.size > 0 and not np.all(particle_region == 0) and \
           background_region.size > 0 and not np.all(background_region == 0):
            snr_i = SNR_computation(particle_region, background_region)
            snr_list.append(snr_i)

    N = len(snr_list)
    snr_val = 10 * np.mean(snr_list) if N > 0 else 0.0

    return snr_val, N

# --- Version 2: Mask-Based Logic ---

def SNR_within_blocks_mask_based(cfg: SNR_config, np_img_data: np.ndarray, mask_data: np.ndarray) -> Tuple[float, int]:
    """
    Calculates the mean SNR using the mask-based pure particle/background selection.
    Includes filtering for non-overlap.
    """

    h, w = np_img_data.shape
    block_size = cfg.block_size

    # 1. Define the valid sampling range based on block size
    if block_size % 2 == 0:
        half_floor = block_size // 2 - 1
    else:
        half_floor = block_size // 2

    valid_x_range = range(cfg.border_size + half_floor, w - cfg.border_size - half_floor)
    valid_y_range = range(cfg.border_size + half_floor, h - cfg.border_size - half_floor)

    particle_centers_all = []
    background_centers_all = []

    # Iterate over all possible block centers
    for x in valid_x_range:
        for y in valid_y_range:
            try:
                print(f"test : \n mask data: {mask_data}, \n x: {x}, \n y : {y}, \n size: {block_size}")
                mask_block = _extract_block(mask_data, x, y, block_size)
            except ValueError:
                continue

            # Check for pure particle block (all pixels must be 1.0)
            if np.all(np.isclose(mask_block, 1.0)):
                particle_centers_all.append((x, y))

            # Check for pure background block (all pixels must be 0.0)
            elif np.all(np.isclose(mask_block, 0.0)):
                background_centers_all.append((x, y))


    # 2. Filter for Non-Overlap on both particle and background centers
    rng = np.random.default_rng(cfg.random_state)
    min_center_dist = cfg.block_size + cfg.min_block_distance

    def filter_non_overlap(centers: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
        """Filters a list of centers to ensure no two selected centers are too close."""
        if not centers: return []
        centers = centers.copy()
        rng.shuffle(centers)

        filtered_coords = []
        for x, y in centers:
            if len(filtered_coords) >= cfg.max_sets_to_compute:
                break

            is_too_close = False
            for px, py in filtered_coords:
                dist_x = abs(x - px)
                dist_y = abs(y - py)

                if dist_x < min_center_dist and dist_y < min_center_dist:
                    is_too_close = True
                    break

            if not is_too_close:
                filtered_coords.append((x, y))
        return filtered_coords

    particle_centers_filtered = filter_non_overlap(particle_centers_all)
    background_centers_filtered = filter_non_overlap(background_centers_all)

    current_num_sets = min(len(particle_centers_filtered), len(background_centers_filtered))

    if current_num_sets == 0:
        return 0.0, 0

    particle_centers_sampled = random.sample(particle_centers_filtered, current_num_sets)
    background_centers_sampled = random.sample(background_centers_filtered, current_num_sets)

    # 3. Extract blocks and compute SNR (Numba optimized loop)
    snr_list = []
    for i in range(current_num_sets):
        px, py = particle_centers_sampled[i]
        particle_region = _extract_block(np_img_data, px, py, block_size)

        bx, by = background_centers_sampled[i]
        background_region = _extract_block(np_img_data, bx, by, block_size)

        if particle_region.size > 0 and not np.all(particle_region == 0) and \
           background_region.size > 0 and not np.all(background_region == 0):
            snr_i = SNR_computation(particle_region, background_region)
            snr_list.append(snr_i)

    N = len(snr_list)
    snr_val = 10 * np.mean(snr_list) if N > 0 else 0.0

    return snr_val, N

# --- Multiprocessing Worker Function ---
def _process_single_micrograph(cfg: SNR_config, micrograph_data: Tuple[np.ndarray, str], method: str) -> Tuple[float, int, str]:
    """
    Worker function executed by the multiprocessing pool to process one micrograph.
    Returns: (mean_snr, N, img_filename)
    """
    img_data, img_filename = micrograph_data
    file_stem = Path(img_filename).stem
    micrograph_mean_snr = 0.0
    N = 0

    try:
        if method == 'block_based':
            coor_path = cfg.PARTICLE_COOR_DIR / f"{file_stem}.csv"
            particles_coor = load_particle_coor(cfg, coor_path)
            micrograph_mean_snr, N = SNR_within_blocks_block_based(cfg, img_data, particles_coor)

        elif method == 'mask_based':
            mask_path = cfg.MASK_DIR / f"{file_stem}.png"
            mask_data = load_mask(cfg, mask_path)
            micrograph_mean_snr, N = SNR_within_blocks_mask_based(cfg, img_data, mask_data)

        # Write CSV result immediately from the worker process
        if N > 0:
            SNR_csv_write_micrograph_data(
                cfg=cfg,
                mrc_filename=img_filename,
                method=method,
                mean_micrograph_snr=micrograph_mean_snr,
                total_pairs_N=N,
            )

    except Exception as e:
        print(f"CRITICAL ERROR in worker for {img_filename}: {e}")
        micrograph_mean_snr = 0.0
        N = 0

    return micrograph_mean_snr, N, img_filename

# --- Main Class and Orchestration ---

class SNRCalculator:
    """
    Main class to configure, run, and aggregate the parallel SNR calculation.
    """
    def __init__(self, config: SNR_config = SNR_config):
        self.data_list = []
        if config.max_sets_to_compute < 1:
            raise ValueError(f"max_sets_to_compute must be at least 1 for mean calculation. Got {config.max_sets_to_compute}")

    def calculate_final_stats(self, config: SNR_config = SNR_config):
        """
        Computes and prints final statistics (CI, distribution data)
        based on the collected micrograph mean SNRs (config.SNR_list).
        """
        samples = np.array(config.SNR_list)
        N = len(samples)

        if N < 2:
            print("\n--- Final Statistics (Skipped) ---")
            print(f"Need at least 2 micrograph mean SNRs to compute CI/Box Plot data. Found N={N}.")
            print("---------------------------------")
            return

        print("\n" + "=" * 40)
        print("Final Statistical Summary (Across all Micrographs)")
        print("=" * 40)

        # 1. 95% Confidence Interval (t-distribution)
        mean_all = np.mean(samples)
        std_err = stats.sem(samples)
        degrees_freedom = N - 1

        if std_err == 0:
            ci_lower = ci_upper = mean_all
        else:
            ci_margin = std_err * t.ppf(0.975, degrees_freedom)
            ci_lower = mean_all - ci_margin
            ci_upper = mean_all + ci_margin

        print(f"Total Micrographs (N): {N}")
        print(f"Overall Mean SNR: {mean_all:.6f}")
        print(f"95% Confidence Interval: ({ci_lower:.4f}, {ci_upper:.4f})")

        # 2. Visualization Data (Box Plot / Density)
        print("\n[Visualization Data for Box/Density Plots]")
        print("Sample distribution statistics (using Micrograph Mean SNRs as samples):")
        print(f"  Median: {np.median(samples):.4f}")
        print(f"  25th Percentile (Q1): {np.percentile(samples, 25):.4f}")
        print(f"  75th Percentile (Q3): {np.percentile(samples, 75):.4f}")
        print("  Full Sample List (for plotting density/box plot):")
        if N > 10:
             print(f"    {samples[:5].tolist()} ... (showing first 5)")
        else:
             print(f"    {samples.tolist()}")
        print("---------------------------------")


    def mean_SNR_within_micrographs(self, data_name: str, method: str = 'block_based', config: SNR_config = SNR_config) -> float:
        """
        Orchestrates parallel processing, collects results, and generates final reports.
        """
        if self.data_list == []:
            self.data_list = load_mrcfile(config)
        n_micrographs = len(self.data_list)
        if n_micrographs == 0:
            return 0.0

        print(f"\nProcessing {n_micrographs} micrographs using {method} method (CPU Parallelized)...")

        config.SNR_list = []

        # --- Multiprocessing Setup ---
        num_workers = os.cpu_count() - 2 if os.cpu_count() > 1 else 1
        print(f"Using {num_workers} CPU workers for parallel processing.")

        worker_func = functools.partial(_process_single_micrograph, config, method=method)

        all_results = []

        # Start multiprocessing pool and wrap the input data in tqdm for progress tracking
        with multiprocessing.Pool(processes=num_workers) as pool:
            # Use pool.imap for lazy evaluation and tqdm wrapping for progress bar
            results_iterator = pool.imap(worker_func, self.data_list)

            # Wrap the iterator with tqdm
            all_results = list(tqdm(results_iterator, total=n_micrographs, desc="Processing Micrographs"))


        # --- Collect Results and Finalize ---

        # 1. Collect results from parallel processes
        for mean_snr, N, img_filename in all_results:
            if N > 0:
                config.SNR_list.append(mean_snr)
                # Note: Worker function prints individual file status, so minimal print here
            else:
                 # Minimal printout for skipped files
                 pass


        # 2. Final Output
        if not config.SNR_list:
             mean_mSNR = 0.0
        else:
             mean_mSNR = np.mean(config.SNR_list)

        self.calculate_final_stats()

        output_path_txt = config.OUTPUT_DIR / "SNR_rst.txt"
        try:
            with open(output_path_txt, 'w') as f:
                f.write(f"Overall Mean Micrograph SNR ({data_name}, {method}): {mean_mSNR:.6f}\n")
                f.write("This is the average of the single mean SNRs calculated for each micrograph.\n")

            print("\n" + "=" * 60)
            print(f"Overall Mean Micrograph SNR: {mean_mSNR:.6f}")
            print(f"Final summary written to {output_path_txt}")
            print("=" * 60)
        except Exception as e:
            print(f"ERROR: Failed to write overall summary to {output_path_txt}. Error: {e}")

        return mean_mSNR

In [16]:
# @title 5. execute
if __name__ == "__main__":
    multiprocessing.set_start_method('spawn', force=True)
    method = "blocks" # @param ["blocks", "mask"]
    method_use = "block_based" if method == "blocks" else "mask_based"

    SNR_run = SNRCalculator()
    SNR_rst = SNR_run.mean_SNR_within_micrographs(data_name= 'test', method= method_use)

Found 84 .mrc files in /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/TpzD_micrographs.
Successfully loaded Falcon_2012_06_12-15_33_42_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-15_36_26_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-15_41_22_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-15_43_48_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-15_46_37_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-15_53_09_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-16_26_22_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-16_44_07_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-16_48_06_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-16_51_03_0.mrc with shape (4096, 4096).
Successfully loaded Falcon_2012_06_12-16_55_40_0.mrc with shape (4096, 4096).
Successf

Processing Micrographs:   0%|          | 0/84 [06:58<?, ?it/s]


KeyboardInterrupt: 